In [1]:
# pip install autolrtuner
!pip install audiomentations

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.1/86.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 9.2 MB/s eta 0:00:00
  Attempting uninstall: soxr
    Found existing installation: soxr 1.0.0
    Uninstalling soxr-1.0.0:
      Successfully uninstalled soxr-1.0.0


# Set up library

In [2]:
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.callbacks import ModelCheckpoint,  EarlyStopping
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Dense, Dropout, Flatten, BatchNormalization
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import LabelEncoder
import h5py
from sklearn.utils.class_weight import compute_class_weight
# from autolrtuner import AutoLRTuner
import os
from audiomentations import Compose, AddGaussianNoise, Gain





# Mount Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Data Agumentations

In [4]:
augment = Compose([

    # Noise Injection
    AddGaussianNoise(
        min_amplitude=0.001,
        max_amplitude=0.01,
        p=0.5
    ),

    # Random Gain
    Gain(
        min_gain_db=-6,
        max_gain_db=6,
        p=0.5
    )

])

#Data Module

In [5]:
from os.path import isfile
class DataModule:

    def __init__(self):
        self.data_set_dic = {}

    def __extract_dft(self, window):
        eps = 1e-10
        spectrum = np.abs(np.fft.rfft(window))
        spectrum = spectrum / np.max(spectrum + eps)
        return spectrum

    def __suppress_low_amplitudes(self, spec, threshold_db=50):
        I_threshold = 10 ** (-threshold_db / 20)
        spec[spec < I_threshold] *= 0.2
        return spec
    def __get_spectrum_of_window(self, window):

        spectrum = self.__extract_dft(window)

        #preprocessing
        standard_spectrum = self.__suppress_low_amplitudes(spectrum)

        return standard_spectrum
    def __get_spectrograms_and_labels_of_each_audio(self, audio, df, sample_rate, samples_per_window):

        X = []
        y = []


        # preparing data
        for _, row in df.iterrows():
            start_sample = int(row["start_time"] * sample_rate)
            end_sample = start_sample + samples_per_window

            if end_sample > len(audio):
                continue

            window = audio[start_sample:end_sample]
            spectrum = self.__get_spectrum_of_window(window)

            X.append(spectrum)
            y.append(row["label"])

        # reducing silence sample
        silence_label = "silence"
        flag = True
        start_index = 0
        end_index = len(y) - 1
        max_silence_label = 3
        X_reduced = []
        y_reduced = []
        while flag:
          if y[start_index] == silence_label:
                start_index += 1

          if y[end_index] == silence_label:
                end_index -= 1

          if y[start_index] != silence_label and y[end_index] != silence_label:
                X_reduced = X[start_index:end_index + 1]
                y_reduced = y[start_index:end_index + 1]


                num_start_silence = start_index - 0
                num_end_silence = len(y) - 1 - end_index

                if num_start_silence > max_silence_label:
                    num_start_silence = max_silence_label

                if num_end_silence > max_silence_label:
                    num_end_silence = max_silence_label

                for i in np.arange(num_start_silence):
                    X_reduced.append(X[i])
                    y_reduced.append(y[i])

                for i in np.arange(num_end_silence):
                    end_index = len(y) - 1 - i
                    X_reduced.append(X[end_index])
                    y_reduced.append(y[end_index])

                flag = False


        return np.array(X_reduced), np.array(y_reduced)


    def process_one_file_audio(
        self,
        audio_path,
        csv_path,
        sample_rate=48000,
        window_size=0.01
    ):

        #load audio
        audios = []
        audio, sr = librosa.load(audio_path, sr=sample_rate, mono=True)
        audios.append(audio)

        #data augmentations
        quantities_au = 2
        for i in range(quantities_au):
            augmented_audio = augment(samples=audio, sample_rate=sr)
            audios.append(augmented_audio)

        samples_per_window = int(sample_rate * window_size)

        # Load CSV
        df = pd.read_csv(csv_path)
        X = []
        y = []

        # preparing data
        for audio in audios:
            X_temp, y_temp = self.__get_spectrograms_and_labels_of_each_audio(audio, df, sample_rate, samples_per_window)
            X.append(X_temp)
            y.append(y_temp)
        return X, y

    def build_dataset(self):

        # read data from drive
        audio_data_set_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/Data/audio"
        audio_folder_name = "audio"
        labels_folder_name = "labels"

        if os.path.exists(audio_data_set_path):
          for root, dirs, files in os.walk(audio_data_set_path):
            for file in files:
              if file.endswith(".wav"):
                audio_file_path = os.path.join(root, file)
                labels_file_name = file.replace(".wav", ".csv")
                labels_file_root = root.replace(audio_folder_name, labels_folder_name)
                #check if specified audio has corresponded labels file
                labels_path = os.path.join(labels_file_root, labels_file_name)
                # print("check audio path and labels path: ", audio_file_path, labels_path)
                if os.path.exists(labels_path):
                    if os.path.exists(labels_path):
                      self.data_set_dic[audio_file_path] = labels_path
                    else:
                      print(labels_path, "is not a file")
                else:
                  print("Cant find file at", labels_path)

        else:
          print("Cant find folder at /content/drive/MyDrive/Qeej_Hmong_Dataset/Data/audio")

        X_all = []
        y_all = []

        for audio_file_path, labels_file_path in self.data_set_dic.items():
          #  print ("check audio path and corresponding label path: ", audio_file_path, labels_file_path)
           X, y = self.process_one_file_audio(audio_file_path, labels_file_path)
           X_all.extend(X)
           y_all.extend(y)

        X_all = np.vstack(X_all)
        y_all = np.hstack(y_all)

        return X_all, y_all

#Main

## Set up HDF5 File

In [6]:
HDF5_file_path = "/content/drive/MyDrive/Qeej_Hmong_Dataset/HDF5"
name_file_save = "Qeej_Hmong_Features_With_Data_Au(repeat = 2).h5"
if os.path.exists(HDF5_file_path):
        print("set up data")
        dataModule = DataModule()
        X, y = dataModule.build_dataset()
        print("check len X:", len(X))
        print("check len y:", len(y))
        data_set_path = os.path.join(HDF5_file_path, name_file_save)
        with h5py.File(data_set_path, "w") as f:
          f.create_dataset("features", data=X)
          f.create_dataset("labels", data=y.tolist())

set up data
Cant find file at /content/drive/MyDrive/Qeej_Hmong_Dataset/Data/labels/Giai điệu/Khèn 1/ntiv.csv
Cant find file at /content/drive/MyDrive/Qeej_Hmong_Dataset/Data/labels/Giai điệu/Khèn 1/ntiv_4.csv
Cant find file at /content/drive/MyDrive/Qeej_Hmong_Dataset/Data/labels/Giai điệu/Khèn 1/ntiv_2.csv


KeyboardInterrupt: 